# Posner / IBM Readiness Evidence

This notebook verifies the local readiness gates for publication-grade Posner verification before any IBM QPU spend.

## Evidence Boundary

This notebook is a readiness and refusal-path demonstration only. It does not claim that ORCA-derived `hf.json`, a complete `extended.json`, IBM calibration data, or QPU results are already available. It checks that incomplete Posner runtime data is rejected, records which local acquisition artifacts are present, and estimates the minimum planned QPU shot budget from the current submission script shape.

In [ ]:
from __future__ import annotations

import argparse
import json
import re
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace

REPO = Path.cwd()
if not (REPO / "tools").exists():
    REPO = Path.cwd().parent
TOOLS = REPO / "tools"
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

POSNER_DIR = REPO / "results" / "posner_external_data"
README = POSNER_DIR / "README.md"
PARSED = POSNER_DIR / "parsed"
IBM_DIR = POSNER_DIR / "ibm"

readme_text = README.read_text(encoding="utf-8")
active_job = re.search(r"projects/\d+/locations/[^/]+/customJobs/\d+", readme_text)
active_state = re.search(r"Last checked state on [^:]+: `([^`]+)`", readme_text)

local_artifacts = {
    "hf_json": PARSED / "hf.json",
    "extended_json": PARSED / "extended.json",
    "extended_partial_json": PARSED / "extended.partial.json",
}
artifact_status = {name: path.exists() for name, path in local_artifacts.items()}
ibm_calibration_files = sorted(IBM_DIR.glob("*_calibration_*.json")) if IBM_DIR.exists() else []

acquisition_status = {
    "active_vertex_job": active_job.group(0) if active_job else None,
    "active_vertex_state": active_state.group(1) if active_state else None,
    "local_artifacts": artifact_status,
    "ibm_calibration_snapshots": [str(path.relative_to(REPO)) for path in ibm_calibration_files],
}
acquisition_status

In [ ]:
from acquire_posner_external_data import validate_runtime

required_extended_fields = {
    "ca43_hf_tensors",
    "ca_electron_map",
    "incorporation_tensors",
    "nuclear_dipolar_pairs",
    "transport_depolarizing_rates",
    "cage_dephasing_rate",
}

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    hf_path = tmp_path / "hf.json"
    extended_path = tmp_path / "extended.json"
    hf_path.write_text(json.dumps({"site1": [], "site2": []}), encoding="utf-8")
    extended_path.write_text(json.dumps({}), encoding="utf-8")

    try:
        validate_runtime(argparse.Namespace(hf_json=str(hf_path), extended_json=str(extended_path)))
    except SystemExit as exc:
        runtime_validation_refusal = str(exc)
    else:
        raise AssertionError("incomplete Posner runtime JSON was accepted")

assert "Missing runtime fields" in runtime_validation_refusal
for field in required_extended_fields:
    assert f"extended.{field}" in runtime_validation_refusal

runtime_validation_refusal

In [ ]:
from verify_ibm_heron import _load_runtime_parameters

missing_hf_args = SimpleNamespace(
    hf1=None,
    hf2=None,
    hf_json=None,
    extended_params=None,
    extended_json=None,
)
try:
    _load_runtime_parameters(missing_hf_args, require_extended=True)
except ValueError as exc:
    missing_hf_refusal = str(exc)
else:
    raise AssertionError("publication-grade path accepted missing hf-json")

hf_stub = [{"Axx": 0.0, "Ayy": 0.0, "Azz": 0.0, "Axy": 0.0, "Axz": 0.0, "Ayz": 0.0} for _ in range(3)]
missing_extended_args = SimpleNamespace(
    hf1=hf_stub,
    hf2=hf_stub,
    hf_json=None,
    extended_params=None,
    extended_json=None,
)
try:
    _load_runtime_parameters(missing_extended_args, require_extended=True)
except ValueError as exc:
    missing_extended_refusal = str(exc)
else:
    raise AssertionError("publication-grade path accepted missing extended-json")

assert "--hf-json is required" in missing_hf_refusal
assert "--extended-json with nuclear_dipolar_pairs is required" in missing_extended_refusal
runtime_gate_refusals = {
    "missing_hf_json": missing_hf_refusal,
    "missing_extended_json": missing_extended_refusal,
}
runtime_gate_refusals

In [ ]:
planned_submit_only_circuits = [
    "exchange_J1",
    "exchange_J10",
    "decoherence_raw",
    "decoherence_xy4",
    "chain_10q",
]
default_shots = 4096
qpu_budget_estimate = {
    "circuit_count": len(planned_submit_only_circuits),
    "shots_per_circuit": default_shots,
    "minimum_shot_circuits": len(planned_submit_only_circuits) * default_shots,
    "excludes": [
        "IBM queue time",
        "calibration acquisition",
        "transpilation retries",
        "repeat batches needed after molecular-data review",
    ],
    "source": "tools/verify_ibm_heron.py _submit_only circuit list and default --shots",
}
qpu_budget_estimate

In [ ]:
gates = {
    "orca_hf_json_present": artifact_status["hf_json"],
    "complete_extended_json_present": artifact_status["extended_json"],
    "ibm_calibration_present": bool(ibm_calibration_files),
    "runtime_validator_rejects_incomplete_json": "Missing runtime fields" in runtime_validation_refusal,
    "verification_runner_requires_hf_json": "--hf-json is required" in missing_hf_refusal,
    "verification_runner_requires_extended_json_for_submission": "--extended-json with nuclear_dipolar_pairs is required" in missing_extended_refusal,
}
ready_for_publication_qpu_submission = all(
    gates[key]
    for key in (
        "orca_hf_json_present",
        "complete_extended_json_present",
        "ibm_calibration_present",
        "runtime_validator_rejects_incomplete_json",
        "verification_runner_requires_hf_json",
        "verification_runner_requires_extended_json_for_submission",
    )
)

manifest = {
    "schema_version": "sc-neurocore.posner-ibm-readiness-evidence.v1",
    "acquisition_status": acquisition_status,
    "gates": gates,
    "ready_for_publication_qpu_submission": ready_for_publication_qpu_submission,
    "qpu_budget_estimate": qpu_budget_estimate,
    "evidence_boundary": "Readiness gates only; no ORCA-derived runtime parameter or IBM QPU-result claim.",
}
manifest